# Web Scraping Notebook (AutoScout24 Example)

This notebook is written for students who are scraping a website for the first time.
Every step is small and explicit, and each code cell has comments that explain exactly what is happening.

What is web scraping?
- A web page is text written in HTML (HyperText Markup Language).
- HTML is made of tags (like <title> or <article>) and attributes (like class or data-*).
- Scraping means downloading that HTML and extracting the parts you need.
- We are NOT using a browser here, only a simple HTTP request.

Key ideas you will learn:
- How to request a page with Python (requests).
- How to read HTML with BeautifulSoup.
- How to extract data from HTML attributes like data-*
- How to store results in a table and save to CSV.

Very important for beginners:
- Always check a website's rules and robots.txt before scraping.
- Scraping too fast can overload a website. Always add delays.
- Some websites load data with JavaScript. If so, requests may show empty pages.
- This notebook is for learning. Use it responsibly and legally.

What we will extract from each listing:
- make
- model
- mileage
- price
- registration date
- fuel type
- country

Important notes for beginners:
- Always check the website rules and robots.txt before scraping.
- Start with a small number of pages while you learn.
- If a page is blocked or requires JavaScript, you may not see the listings.



## Step 0: Install the libraries (only once)

We use three Python libraries:
- `requests` to download web pages
- `beautifulsoup4` to parse HTML
- `pandas` to save data into a table

If you already installed them, you can skip running this cell next time.

What is `pip`?
- `pip` is the package installer for Python.
- It downloads and installs libraries so you can import them in your code.
- In notebooks, `%pip` is a safe way to run pip inside the current environment.



In [1]:
# %pip install requests beautifulsoup4 pandas


## Step 1: Import the libraries

We now import the libraries we just installed. This makes their functions available in the notebook.
Each import is commented so you remember what it is used for.

A quick reminder for beginners:
- `import` lets you use code written by other people.
- Libraries are just collections of useful functions.
- You only need to install a library once, but you must import it every time you run the notebook.



In [2]:
import requests  # Download web pages with HTTP requests
from bs4 import BeautifulSoup  # Parse HTML into searchable elements
import re  # Regex for parsing power values
import pandas as pd  # Store and export data as a table
import time  # Pause between requests to be polite


## Step 2: Define the search parameters

We choose a small list of brands and a short page range. This is enough to learn the process
and keeps the load on the website low. We also create a base URL and a simple User-Agent header
so the request looks like it comes from a normal browser.

Basic URL concept:
- A URL can include query parameters after a `?`.
- Example: `?page=2` means the website should show page 2 of results.
- We will build these URLs in code instead of typing them manually.

Why a User-Agent header?
- Some websites block requests that do not look like a browser.
- A simple User-Agent makes our request more realistic.



In [3]:
make = 'audi'  # Single make to query
model = 'a3'  # Single model to query
country_code = 'D'  # Country code for Germany
country_name = 'germany'  # Used in output filename
start_page = 1  # First results page to request
end_page = 200  # Last page (inclusive) to maximize coverage
base_url = f'https://www.autoscout24.com/lst/{make}/{model}?atype=C&cy={country_code}'  # URL template
headers = {
    'User-Agent': 'Mozilla/5.0 (educational example)'  # Simple browser-like header
}  # Some sites reject requests without a User-Agent


## Step 3: Build one example URL

Before looping over many pages, we build one example URL to verify the format.
This makes debugging easier because we can test a single page first.

Tip: Always test one URL by hand before scaling up to many pages.



In [4]:
example_url = base_url + '&desc=0&page=1'  # Single test URL
example_url  # Display the URL to verify it looks correct


'https://www.autoscout24.com/lst/audi/a3?atype=C&cy=D&desc=0&page=1'

## Step 4: Download the example page

We download the page using a `Session`, which keeps cookies and headers between requests.
We print the HTTP status code and stop if the page did not load correctly.

HTTP basics:
- Status code 200 means OK (the page loaded).
- 403 or 429 often mean blocked or too many requests.
- A timeout prevents your code from waiting forever if the site is slow.



In [5]:
session = requests.Session()  # Reuse cookies/headers across requests
response = session.get(example_url, headers=headers, timeout=10)  # Download the page (10s timeout)
status_code = response.status_code  # HTTP status code returned by the server
print('Status code:', status_code)  # Show the result
if status_code != 200:  # Stop if the response is not OK
    # Raising an error here prevents us from parsing a broken page
    raise ValueError(f'Request failed with status code {status_code}')  # Explain the error


Status code: 200


## Step 5: Parse the HTML

The response is raw HTML text. We convert it into a BeautifulSoup object so we can search it
for specific elements like listing cards.

What is BeautifulSoup?
- It turns HTML text into a tree of elements you can search.
- You can find tags by name, class, or attribute.
- We will use CSS selectors to find listing cards.
- Example selector: `article[data-testid="decluttered-list-item"]` means an article tag with that attribute.



In [6]:
html_text = response.text  # Raw HTML as text
soup = BeautifulSoup(html_text, 'html.parser')  # Parse with Python's built-in HTML parser


## Step 6: Check the page title

We read the `<title>` tag. If the title looks correct, the page probably loaded.
If it is missing or generic, the website might be blocking us or loading data with JavaScript.

This is a quick sanity check before scraping many pages.



In [7]:
title_tag = soup.find('title')  # Look for the title tag
page_title = title_tag.get_text(strip=True) if title_tag else 'No title found'  # Safe extraction
print('Page title:', page_title)  # Display the title


Page title: Used Audi A3 for sale - AutoScout24


## Step 7: Inspect one listing card (checkpoint)

Before scraping many pages, we inspect one listing card to learn the HTML structure.
We print a short HTML snippet so you can see where the data is stored.

Why inspect the HTML?
- Websites change often. This step shows the current structure.
- We look for `data-*` attributes because they often store clean values.



In [8]:
# Checkpoint: inspect one listing card (HTML snippet)
sample_card = soup.select_one('article[data-testid="decluttered-list-item"]')  # Tag with attribute
if sample_card:
    print(sample_card.prettify()[:1500])  # Show the first 1500 chars
    # Example of reading a data-* attribute directly
    print('Listing country code:', sample_card.get('data-listing-country'))
else:
    print('No listing card found on the page.')


<article class="cldt-summary-full-item listing-impressions-tracking list-page-item DeclutteredListItem_wrapper__kZxpf" data-applied-tier="t50" data-applied_boost_level="t50" data-boost_level="t50" data-boosting_product="mia" data-customer-id="3885825" data-deliverable="false" data-first-registration="06-2003" data-fuel-type="d" data-guid="dbedbdbd-925a-4d3d-a1ec-a26e66053a3a" data-image-content="no-placeholder|0.14100863100587757" data-is-smyle-eligible="false" data-leads-range="some" data-listing-country="d" data-listing-zip-code="35394" data-make="audi" data-mia-level="t50" data-mileage="234163" data-model="a3" data-model-taxonomy="[make_id:9, model_group_id:1624, variant_id:151, generation_id:152, motortype_id:240, trim_id:131];" data-order-bucket="0" data-otp="nfm" data-ownership-models="tr" data-position="1" data-price="690" data-price-label="toolow-price " data-relevance_adjustment="sponsored" data-seller-type="d" data-source="listpage_search-results" data-testid="decluttered-lis

## Step 8: Extract data from all pages

We define a small helper function that reads values from the HTML `data-*` attributes on each card.
Then we loop through the brands and pages, collect all listings, and build a table.
We also add short pauses between requests to be polite.

Beginner tips:
- A function is a reusable block of code.
- A list (like `results`) stores many rows before we build a table.
- A DataFrame is a table with rows and columns, like a spreadsheet.
- We sleep between requests to avoid hitting the server too fast.



In [9]:
def extract_power_hp(card):
    # Parse text like "103 kW (140 hp)" and return hp as a string
    for span in card.select("span"):
        text = span.get_text(" ", strip=True)
        match = re.search(r"\((\d+)\s*hp\)", text)
        if match:
            return match.group(1)
    return None

def parse_listing_card(card):
    # Read data directly from HTML data-* attributes on the card
    # data-* attributes are often clean and easier to parse than messy text
    return {
        "make": card.get("data-make"),  # Brand name
        "model": card.get("data-model"),  # Model name
        "mileage": card.get("data-mileage"),  # Mileage in km (as string)
        "price": card.get("data-price"),  # Price in EUR (as string)
        "price_label": card.get("data-price-label"),  # Price label (e.g., good-price)
        "registration": card.get("data-first-registration"),  # First registration date
        "fuel": card.get("data-fuel-type"),  # Fuel type code
        "country": card.get("data-listing-country"),  # Listing country code (not full name)
        "power_hp": extract_power_hp(card),  # Power in hp (as string)
    }

results = []  # List where we will store results (each item is one row)
for page in range(start_page, end_page + 1):  # Iterate over each page
    url = base_url + f'&desc=0&page={page}'  # Build the URL
    response = session.get(url, headers=headers, timeout=10)  # Download the page
    if response.status_code != 200:  # Skip if the request fails
        print(f'Error {response.status_code} on {url}')  # Simple warning
        continue  # Move to the next page
    soup = BeautifulSoup(response.text, 'html.parser')  # Parse the HTML
    # Select all listing cards using a CSS selector
    cards = soup.select('article[data-testid="decluttered-list-item"]')  # Listing cards
    if not cards:
        print(f'No listings found on {url} (page may be blocked or rendered with JS).')
        continue
    for card in cards:
        row = parse_listing_card(card)  # Extract fields from one card
        row['make'] = make  # Add make context
        row['model'] = model  # Add model context
        row['country_name'] = country_name  # Add country context
        row['page'] = page  # Add page context
        results.append(row)  # Append to results
    time.sleep(1)  # Pause to avoid overloading the server
results_df = pd.DataFrame(results)  # Convert to a DataFrame
results_df.head()  # Show the first rows


,make,model,mileage,price,price_label,registration,fuel,country,power_hp,country_name,page
0,audi,a3,234163,690,toolow-price,06-2003,d,d,105,germany,1
1,audi,a3,76000,16980,good-price,12-2014,b,d,179,germany,1
2,audi,a3,174549,9950,top-price,04-2015,d,d,110,germany,1
3,audi,a3,91700,13950,top-price,12-2013,b,d,179,germany,1
4,audi,a3,36216,22950,top-price,12-2022,b,d,150,germany,1


## Step 9: Save the results to the repo data folder

We save the DataFrame into the `data/raw` folder at the root of the repository.
We include a timestamp in the filename so each run creates a new file.

Why save to CSV?
- CSV is a simple text format that works in Excel, Google Sheets, and Python.
- Saving to disk means you do not lose data if the notebook restarts.



In [10]:
from datetime import datetime
from pathlib import Path

# Resolve repo root (walk up until we find the data folder)
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

output_dir = repo_root / 'data' / 'raw'  # Output directory at repo root
output_dir.mkdir(parents=True, exist_ok=True)  # Create if missing
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')  # Versioned filename
output_path = output_dir / f'autoscout24_listings_{make}_{model}_{country_name}_{timestamp}.csv'  # Output file path
results_df.to_csv(output_path, index=False)  # Save without the index column
print('Saved to', output_path)  # Confirmation message


Saved to c:\Users\cfuen\OneDrive\Documentos\Clases\Ficheros para clase\Albert\Head of Data 101\Github repo\HeadOfData101\data\raw\autoscout24_listings_audi_a3_germany_20251227_230815.csv
